In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import shap, warnings
warnings.filterwarnings("ignore")

# ---------- 0. 路径 ----------
DATA_FILE   = "Identity_scored.xlsx"
UNSCORED    = "images.xlsx"
OUT_DIR     = Path("D:\Desktop\output_Identity2")
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Identity.xlsx"
BSWARM_PNG   = OUT_DIR / "shap_beeswarm_Identity.png"
BAR_PNG      = OUT_DIR / "shap_bar_Identity.png"

# ---------- 1. 常量 ----------
TARGET = "Identity"
FEATS  = ['Road','Building','Pole Group','Indicator','Vegetation','Sky',
          'Person','Car','Motorcycle','Bicycle','Clothes','Trash Can',
          'Riverway','Signboard','Air Conditioner Condenser','Festival Elements']

# ---------- 2. 读取数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)

X = df[FEATS].fillna(0.0)    
y = df[TARGET].values

# ---------- 3. 划分 & 训练 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, random_state=42, shuffle=True)

rf = RandomForestRegressor(
        n_estimators     = 556,
        max_depth        = 17,
        min_samples_leaf = 2,
        max_features     = 0.6798872582107739,
        random_state     = 42,
        n_jobs           = -1
     ).fit(X_tr, y_tr)

# ---------- 4. 评估 ----------
def metric(t, p): return r2_score(t, p), mean_squared_error(t, p, squared=False)
r2_tr, rmse_tr = metric(y_tr, rf.predict(X_tr))
r2_te, rmse_te = metric(y_te, rf.predict(X_te))
r2_all, rmse_all = metric(y,     (rf.predict(X)))

print(f"Train R²={r2_tr:.3f} | RMSE={rmse_tr:.3f}")
print(f"Test  R²={r2_te:.3f} | RMSE={rmse_te :.3f}")
print(f"Overall R² = {r2_all:.3f} | RMSE = {rmse_all:.3}")


# ---------- 5. SHAP ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS, show=False)
plt.title("SHAP Beeswarm – Identity")
plt.tight_layout(); plt.savefig(BSWARM_PNG, bbox_inches="tight",dpi=300); plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS,
                  plot_type="bar", show=False)
plt.title("Feature Importance – Identity")

plt.tight_layout(); plt.savefig(BAR_PNG, bbox_inches="tight",dpi=300); plt.close()

# ---------- 6. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
X_new  = df_new[FEATS].fillna(0.0)
df_new[TARGET] = rf.predict(X_new).round(5)
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测文件:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)


Train R²=0.974 | RMSE=0.086
Test  R²=0.890 | RMSE=0.181
Overall R² = 0.948 | RMSE = 0.123
✔ 预测文件: D:\Desktop\output_Identity2\images_predicted_Identity.xlsx
✔ SHAP 图: D:\Desktop\output_Identity2\shap_beeswarm_Identity.png D:\Desktop\output_Identity2\shap_bar_Identity.png
